# LongiControl: your first trained agent

Run this notebook from top to bottom to load a bundled Stable-Baselines3 SAC agent, inspect a drive, and compare it with a random policy. **Run All does not train, download models, or write output files.** The small CPU demo is not an optimized or safety-validated controller.

From the repository root, install and launch with the same Python environment:

```bash
python -m pip install -e ".[examples]"
python -m jupyterlab examples/quickstart.ipynb
```

If you select a different notebook kernel, install the extra in that kernel's environment too.

In [ ]:
%matplotlib inline

from pathlib import Path
import json
import sys

# Jupyter may start in either the checkout root or examples/.
root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "examples/sb3_quickstart.py").is_file()),
    None,
)
if root is None:
    raise RuntimeError("Start Jupyter inside the LongiControl checkout.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import matplotlib.pyplot as plt
import torch
from stable_baselines3.common.env_checker import check_env
from examples.sb3_quickstart import (
    DEFAULT_SEEDS, compare, load_model, make_env, plot_rollout, rollout,
)

# A tiny network is faster and more reproducible with one CPU thread.
torch.set_num_threads(1)
print(f"Python: {sys.version.split()[0]}")

## 1. Load the bundled demonstration

The model was trained on `StochasticTrack-v1` with the unchanged v1 reward. Its metadata records the training settings, software versions, and SHA-256. Only load SB3 archives from sources you trust: SB3 can deserialize pickle-based metadata. A matching checksum detects corruption; it does not make an untrusted model safe.

In [ ]:
model, metadata = load_model()
print(json.dumps({key: metadata[key] for key in (
    "env_id", "training_seed", "training_steps", "versions", "purpose",
)}, indent=2))

## 2. Check the Gymnasium/SB3 interface

The observation contains eight normalized values. The action is a one-element acceleration-control array in `[-1, 1]`. Always stop on either `terminated` (finish reached) or `truncated` (time limit).

In [ ]:
env = make_env(metadata["env_id"])
try:
    check_env(env.unwrapped)
    observation, info = env.reset(seed=DEFAULT_SEEDS[0])
    action, _ = model.predict(observation, deterministic=True)
    print("Observation:", observation)
    print("Action:", action)
    _, reward, terminated, truncated, info = env.step(action)
    print("Reward components:", info["reward_components"])
finally:
    env.close()

## 3. Inspect one complete drive

The plots show speed-limit compliance, acceleration, and **model-estimated net energy**, including negative power predictions during recuperation. The model has no traffic participants or collision checking.

In [ ]:
drive = rollout(model, env_id=metadata["env_id"], seed=DEFAULT_SEEDS[0])
print(json.dumps(drive["metrics"], indent=2))
figure = plot_rollout(drive)
plt.show()
plt.close(figure)

## 4. Compare on the same five tracks

Both policies start fresh environments with the same evaluation seeds. These are separate from the training seed, but this tiny example is **not** a held-out benchmark or a multiple-training-seed study. All means include timeouts: inspect completion rate and distance before interpreting travel time or energy. A policy that stops early is not necessarily efficient.

In [ ]:
comparison = compare(model, env_id=metadata["env_id"], seeds=DEFAULT_SEEDS)
print(json.dumps({name: comparison[name]["summary"] for name in ("sac", "random")}, indent=2))

## 5. Train your own agent (optional)

Run these commands in a terminal **from the checkout root**; they are not executed by this notebook. Training uses one CPU thread, a fixed seed and a small network. Start with 2,000 steps to check the workflow; this does not imply useful driving behavior.

```bash
python -m examples.sb3_quickstart train --steps 2000 --output runs/my-first-sb3
python -m examples.sb3_quickstart demo --model runs/my-first-sb3/model.zip
```

Use 20,000 steps and seed 42 to reproduce the bundled training configuration. Use a new output directory for each run: existing runs are never overwritten. Saved files are `model.zip`, `metadata.json`, and `evaluation.json`. This example does not save replay/RNG state for exact training continuation; the separate `training` package provides that workflow.

See [the examples guide](README.md) and [the demo model card](models/sac_demo/README.md) for CLI options, recorded results, and limitations.